In [1]:
from pathlib import Path
import runpy

script_candidates = (Path.cwd() / "base.py", Path.cwd() / "3D" / "base.py")
simulation_script = next((path for path in script_candidates if path.is_file()), None)
if simulation_script is None:
    raise FileNotFoundError(f"Could not find base.py in {script_candidates}")

simulation_state = runpy.run_path(str(simulation_script))
simulation_state.pop("__name__", None)
globals().update(simulation_state)


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 13 (Line)
Info    : [ 10%] Meshing curve 14 (Line)
Info    : [ 10%] Meshing curve 15 (Line)
Info    : [ 10%] Meshing curve 16 (Line)
Info    : [ 10%] Meshing curve 17 (Line)
Info    : [ 10%] Meshing curve 18 (Line)
Info    : [ 20%] Meshing curve 19 (Line)
Info    : [ 20%] Meshing curve 20 (Line)
Info    : [ 20%] Meshing curve 21 (Line)
Info    : [ 20%] Meshing curve 22 (Line)
Info    : [ 20%] Meshing curve 23 (Line)
Info    : [ 20%] Meshing curve 24 (Line)
Info    : [ 30%] Meshing curve 25 (Line)
Info    : [ 30%] Meshing curve 26 (Line)
Info    : [ 30%] Meshing curve 27 (Line)
Info    : [ 30%] Meshing curve 28 (Line)
Info    : [ 30%] Meshing curve 29 (Line)
Info    : [ 30%] Meshing curve 30 (Line)
Info    : [ 40%] Meshing curve 31 (Line)
Info    : [ 40%] Meshing curve 32 (Line)
Info    : [ 40%] Meshing curve 33 (Line)
Info    : [ 40%] Meshing curve 34 (Line)
Info    : [ 40%] Meshing curve 35 (Line)
Info    : [ 40%] Meshing curve 36

In [ ]:
# Interactive PyVista view: run the simulation cell above first.
import ipywidgets as widgets
import pyvista as pv
from dolfinx import plot
from IPython.display import clear_output, display

if domain.comm.size != 1:
    raise RuntimeError("The interactive PyVista viewer requires a serial run; use the VTX files for MPI output.")

pv.set_jupyter_backend("static")
tdim = domain.topology.dim
topology, cell_types, points = plot.vtk_mesh(domain, tdim)
trajectory_grid = pv.UnstructuredGrid(topology, cell_types, points)
velocity_space = fem.functionspace(domain, ("DG", 0, (domain.geometry.dim,)))

field_frames = {
    "density": [function.x.array.real.copy() for function in m_traj],
    "value_function": [function.x.array.real.copy() for function in u_traj],
}
velocity_frames = []
for value in u_traj:
    velocity = fem.Function(velocity_space)
    velocity_expression = fem.Expression(
        -ufl.grad(value), velocity_space.element.interpolation_points
    )
    velocity.interpolate(velocity_expression)
    velocity.x.scatter_forward()
    velocity_frames.append(velocity.x.array.real.reshape((-1, domain.geometry.dim)).copy())

for field_name, frames in field_frames.items():
    if any(not np.isfinite(frame).all() for frame in frames):
        raise FloatingPointError(f"Cannot visualize non-finite {field_name} values")
if any(not np.isfinite(frame).all() for frame in velocity_frames):
    raise FloatingPointError("Cannot visualize non-finite velocity values")
if any(frame.size != trajectory_grid.n_cells for frame in field_frames["density"]):
    raise ValueError("DG0 density data must contain one value per air-mesh cell.")
if any(frame.size != trajectory_grid.n_points for frame in field_frames["value_function"]):
    raise ValueError("P1 value data must contain one value per air-mesh point.")
if any(frame.shape != (trajectory_grid.n_cells, domain.geometry.dim) for frame in velocity_frames):
    raise ValueError("DG0 velocity data must contain one vector per air-mesh cell.")
field_ranges = {
    field_name: (
        min(float(frame.min()) for frame in frames),
        max(float(frame.max()) for frame in frames),
    )
    for field_name, frames in field_frames.items()
}

building_bounds = [
    (15, 35, 15, 35, 0, 35),
    (60, 85, 15, 30, 0, 40),
    (15, 30, 60, 85, 0, 30),
    (55, 80, 55, 80, 0, 45),
]
z_slice = 10.0
trajectory_grid.cell_data["density"] = field_frames["density"][0]
trajectory_grid.cell_data["velocity"] = velocity_frames[0]
trajectory_grid.point_data["value_function"] = field_frames["value_function"][0]


def make_slice(frame, field, slice_z):
    trajectory_grid.cell_data["density"] = field_frames["density"][frame]
    trajectory_grid.cell_data["velocity"] = velocity_frames[frame]
    trajectory_grid.point_data["value_function"] = field_frames["value_function"][frame]
    result = trajectory_grid.slice(normal="z", origin=(50, 50, slice_z))
    result = result.cell_data_to_point_data(pass_cell_data=True)
    result.point_data["trajectory"] = result.point_data[field]
    return result


slice_mesh = make_slice(0, "density", z_slice)
plotter = pv.Plotter(window_size=(1000, 700), off_screen=True)
slice_actor = plotter.add_mesh(
    slice_mesh,
    scalars="trajectory",
    cmap="viridis",
    opacity=0.95,
    scalar_bar_args={"title": "Density"},
)
flow_actor = plotter.add_mesh(
    slice_mesh.glyph(orient="velocity", scale=False, factor=2.0),
    color="white",
    name="flow_velocity",
)
plotter.add_mesh(pv.Box(bounds=(0, 100, 0, 100, 0, 50)), style="wireframe", color="black", line_width=2)
for bounds in building_bounds:
    plotter.add_mesh(pv.Box(bounds=bounds), color="dimgray", opacity=0.45)
    plotter.add_mesh(pv.Box(bounds=bounds), style="wireframe", color="white", line_width=2)
plotter.add_mesh(pv.Sphere(radius=2.0, center=start_position), color="limegreen", name="start_marker")
plotter.add_mesh(pv.Sphere(radius=2.0, center=goal_position), color="red", name="goal_marker")
plotter.add_point_labels(
    [start_position, goal_position], ["Start", "Goal"], point_size=0, font_size=14
)
plotter.add_axes()
plotter.show_grid(xtitle="x (m)", ytitle="y (m)", ztitle="z (m)")
plotter.camera_position = "iso"


def density_centroid(density):
    mass_local = fem.assemble_scalar(fem.form(density * ufl.dx))
    mass = domain.comm.allreduce(mass_local, op=MPI.SUM)
    if not np.isfinite(mass) or mass <= 0.0:
        raise ValueError(f"Density has invalid total mass: {mass}")
    moments = [
        domain.comm.allreduce(
            fem.assemble_scalar(fem.form(x_coord[i] * density * ufl.dx)),
            op=MPI.SUM,
        )
        for i in range(3)
    ]
    centroid = np.asarray(moments, dtype=float) / mass
    if not np.isfinite(centroid).all():
        raise FloatingPointError("Density centroid contains non-finite values")
    return centroid


density_centroids = np.asarray([density_centroid(density) for density in m_traj])
initial_goal_distance = np.linalg.norm(density_centroids[0] - goal_position)
final_goal_distance = np.linalg.norm(density_centroids[-1] - goal_position)
print(f"Initial density centroid: {density_centroids[0]}")
print(f"Final density centroid:   {density_centroids[-1]}")
print(f"Distance to goal: {initial_goal_distance:.2f} -> {final_goal_distance:.2f} m")
if final_goal_distance >= initial_goal_distance:
    print("WARNING: the population centroid did not move closer to the goal; inspect solver convergence and parameters.")

frame_slider = widgets.IntSlider(value=0, min=0, max=len(m_traj) - 1, description="Time step", continuous_update=False)
play = widgets.Play(value=0, min=0, max=len(m_traj) - 1, step=1, interval=250, description="Play")
widgets.jslink((play, "value"), (frame_slider, "value"))
z_slider = widgets.FloatSlider(value=z_slice, min=0, max=50, step=1, description="Slice z (m)", continuous_update=False)
field_selector = widgets.Dropdown(
    options=[("Density", "density"), ("Value function", "value_function")],
    value="density",
    description="Field",
)
centroid_marker = plotter.add_mesh(
    pv.Sphere(radius=1.5, center=density_centroids[0]),
    color="orange",
    name="population_centroid",
)


def update_trajectory(frame, field, slice_z):
    updated_slice = make_slice(frame, field, slice_z)
    slice_actor.mapper.SetInputData(updated_slice)
    slice_actor.mapper.scalar_range = field_ranges[field]
    flow_actor.mapper.SetInputData(
        updated_slice.glyph(orient="velocity", scale=False, factor=2.0)
    )
    field_label = "Density" if field == "density" else "Value function"
    plotter.add_text(
        f"{field_label} | t = {frame * dt:.2f} s | z = {slice_z:g} m",
        name="trajectory_title",
        font_size=12,
    )
    centroid_marker.mapper.SetInputData(
        pv.Sphere(radius=1.5, center=density_centroids[frame])
    )
    plotter.render()
    with view_output:
        clear_output(wait=True)
        display(plotter.show(jupyter_backend="static"))


def on_controls_change(change):
    update_trajectory(frame_slider.value, field_selector.value, z_slider.value)


frame_slider.observe(on_controls_change, names="value")
field_selector.observe(on_controls_change, names="value")
z_slider.observe(on_controls_change, names="value")
view_output = widgets.Output()
display(widgets.VBox([widgets.HBox([play, frame_slider]), field_selector, z_slider]), view_output)
update_trajectory(frame_slider.value, field_selector.value, z_slider.value)


2026-10-15 22:30:04.344 (  14.023s) [    7FE5DA75A140]vtkXOpenGLRenderWindow.:1460  WARN| bad X server connection. DISPLAY=


Initial density centroid: [ 9.66101678  9.6777516  10.39776431]
Final density centroid:   [53.42319982 53.64771445 19.80333472]
Distance to goal: 107.53 -> 44.80 m


Output()